In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split

In [2]:
import json
import os

# Вставьте ваш токен вручную
kaggle_token = {"username":"victoriaalanakyan","key":"9ed90d8a4fa3f060b128deae08f17b0d"}

# Сохраняем токен
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_token, f)
!chmod 600 /root/.kaggle/kaggle.json

In [3]:
!kaggle competitions download -c dog-breed-identification

dog-breed-identification.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
import PIL
import torchvision
from PIL import Image
from torchvision.transforms import v2

In [5]:
import pandas as pd
import zipfile

# Загрузка данных
# Путь к ZIP-файлу
zip_path = "dog-breed-identification.zip"
# Папка, куда распаковывать
extract_dir = "files"

# Распаковка
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Архив распакован в {extract_dir}")

Архив распакован в files


In [6]:
from tqdm import tqdm
from os import listdir
import pandas as pd

df = pd.read_csv("files/labels.csv")
df

,id,breed
0,000bec180eb18c7604dcecc8fe0dba07,boston_bull
1,001513dfcb2ffafc82cccf4d8bbaba97,dingo
2,001cdf01b096e06d78e9e5112d419397,pekinese
3,00214f311d5d2247d5dfe4fe24b2303d,bluetick
4,0021f9ceb3235effd7fcde7f7538ed62,golden_retriever
...,...,...
10217,ffd25009d635cfd16e793503ac5edef0,borzoi
10218,ffd3f636f7f379c51ba3648a9ff8254f,dandie_dinmont
10219,ffe2ca6c940cddfee68fa3cc6c63213f,airedale
10220,ffe5f6d8e2bff356e9482a80a6e29aac,miniature_pinscher


In [7]:
headers = list(sorted(df['breed'].unique()))
len(headers)

120

In [8]:
res = pd.read_csv('files/sample_submission.csv')

res

,id,affenpinscher,afghan_hound,african_hunting_dog,airedale,american_staffordshire_terrier,appenzeller,australian_terrier,basenji,basset,...,toy_poodle,toy_terrier,vizsla,walker_hound,weimaraner,welsh_springer_spaniel,west_highland_white_terrier,whippet,wire-haired_fox_terrier,yorkshire_terrier
0,000621fb3cbb32d8935728e48679680e,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
1,00102ee9d8eb90812350685311fe5890,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
2,0012a730dfa437f5f3613fb75efcd4ce,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
3,001510bc8570bbeee98c8d80c8a95ec1,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
4,001a5f3114548acdefa3d4da05474c2e,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10352,ffeda8623d4eee33c6d1156a2ecbfcf8,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
10353,fff1ec9e6e413275984966f745a313b0,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
10354,fff74b59b758bbbf13a5793182a9bbe4,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333
10355,fff7d50d848e8014ac1e9172dc6762a3,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,...,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333,0.008333


In [9]:
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

class Dog(Dataset):
    def __init__(self, labels_path, root_dir, transform=None):
        self.labels_df = pd.read_csv(labels_path)
        self.root_dir = root_dir
        self.transform = v2.Compose([
            v2.ToImage(),
            v2.Resize(320),
            transforms.RandomRotation(degrees=45),
            v2.CenterCrop(256),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        self.label_encoder = LabelEncoder()
        self.labels_df['breed'] = self.label_encoder.fit_transform(self.labels_df['breed'])

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.labels_df.iloc[idx, 0] + '.jpg')
        image = Image.open(img_name)

        label = self.labels_df.iloc[idx, 1]

        if self.transform:
            resized_image = self.transform(image)

        return resized_image, label

In [10]:
dataset = Dog(
    labels_path='files/labels.csv',
    root_dir='files/train')

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset,
                                                            [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, drop_last=True)

In [11]:
dataloaders = {
    'train' : train_loader,
    'val' : val_loader
}

In [12]:
torch.cuda.empty_cache()

In [13]:
model = models.resnet50(pretrained=True)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.BatchNorm1d(256),
    nn.Linear(256, 120)
)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [14]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [15]:
from tqdm import tqdm

def test(model, loader):
    loss_log = []
    acc_log = []
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    for data, target in tqdm(loader, ncols=80):
        data = data.to(device).float()
        target = target.to(device).long()

        # <your code here>
        logits = model(data)
        loss = loss_fn(logits, target)
        loss = loss.long()

        loss_log.append(loss.item())
    return np.mean(loss_log)

def train_epoch(model, optimizer, train_loader):
    loss_log = []
    acc_log = []
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    for data, target in tqdm(train_loader, ncols=80):
        data = data.to(device).float()
        target = target.to(device).long()
        #print(len(data[0]))
        # <your code here>
        optimizer.zero_grad()
        logits = model(data)
        loss = loss_fn(logits, target)
        loss.backward()
        optimizer.step()
        #print(loss)
        loss_log.append(loss.item())
    return loss_log

def train(model, optimizer, n_epochs, train_loader, val_loader, scheduler=None):
    train_loss_log, val_loss_log = [], []

    for epoch in range(n_epochs):
        train_loss = train_epoch(model, optimizer, train_loader)
        val_loss = test(model, val_loader)

        train_loss_log.extend(train_loss)

        val_loss_log.append(val_loss)

        print(f"Epoch {epoch}")
        print(f" train loss: {np.mean(train_loss)}")
        print(f" val loss: {val_loss}\n")

        if scheduler is not None:
            scheduler.step()

    return train_loss_log, val_loss_log

In [16]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[8, 10], gamma=0.1)
tr_loss_log, tr_acc_log, val_loss_log, val_acc_log = train(model, optimizer, 20, train_loader, val_loader, scheduler)

100%|███████████████████████████████████████████| 31/31 [00:26<00:00,  1.18it/s]


Epoch 0
 train loss: 3.544729137045192
 val loss: 2.0



100%|███████████████████████████████████████████| 31/31 [00:25<00:00,  1.20it/s]


Epoch 1
 train loss: 2.124673552400484
 val loss: 1.032258064516129



100%|███████████████████████████████████████████| 31/31 [00:25<00:00,  1.21it/s]


...

100%|███████████████████████████████████████████| 31/31 [00:25<00:00,  1.21it/s]


Epoch 18
 train loss: 0.18079338568871414
 val loss: 0.12903225806451613



100%|███████████████████████████████████████████| 31/31 [00:26<00:00,  1.18it/s]

Epoch 19
 train loss: 0.1796688872763491
 val loss: 0.0967741935483871



ValueError: not enough values to unpack (expected 4, got 2)

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

def create_prediction_loader(csv_path, img_dir, batch_size=32):
    """
    Создает DataLoader для предсказаний по CSV с именами файлов

    Args:
        csv_path: Путь к CSV файлу с колонкой 'image_path'
        img_dir: Папка с изображениями
        batch_size: Размер батча для предсказаний

    Returns:
        DataLoader, возвращающий (images, image_paths)
    """
    class PredictionDataset(Dataset):
        def __init__(self, df, img_dir, transform):
            self.df = df
            self.img_dir = img_dir
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            img_name = self.df.iloc[idx]['id']
            img_path = os.path.join(self.img_dir, img_name+'.jpg')
            img = Image.open(img_path).convert('RGB')

            if self.transform:
                img = self.transform(img)

            return img, img_name

    # Загрузка CSV
    df = pd.read_csv(csv_path)

    # Трансформы для предобученных моделей
    transform = transforms.Compose([
        transforms.Resize(320),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Создание датасета
    dataset = PredictionDataset(df, img_dir, transform)

    # DataLoader для предсказаний
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        collate_fn=lambda batch: (
            torch.stack([item[0] for item in batch]),  # Тензоры изображений
            [item[1] for item in batch]               # Имена файлов
        )
    )


test_loader = create_prediction_loader(
    csv_path="files/sample_submission.csv",
    img_dir="files/test/",
    batch_size=16
)


predictions = []
for images, img_paths in test_loader:
    images = images.to(device)

    with torch.no_grad():
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()

        for file, prob in zip(img_paths, probs):
            predictions.append([file] + list(prob))

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [18]:
len(predictions[0])

121

In [19]:
len(predictions)

10357

In [20]:
columns = ['id'] + headers
df_submission = pd.DataFrame(predictions, columns=columns)
df_submission['id'] = df_submission['id']
df_submission.to_csv('submission.csv', index=False)